In [ ]:
stream = 'primary'
field = 'signal'

In [ ]:
import json
import os
import uuid

import numpy as np
import scrapbook as sb
from tiled.client import from_uri

from lucid_pipelines.notebook import (
    get_input_access_blob,
    get_input_run,
    get_provenance,
)

In [ ]:
input_run = get_input_run()
stream_node = input_run[stream]

try:
    table = stream_node.read()
    column = table[field]
    signal = np.stack([np.asarray(x) for x in column.values])
except (TypeError, KeyError, AttributeError):
    signal = np.asarray(stream_node[field][:])

print(f'passthrough: shape={signal.shape} dtype={signal.dtype}')

In [ ]:
input_blob = get_input_access_blob()
output_tags = json.loads(os.environ.get('LUCID_OUTPUT_TAGS', '[]'))
input_uid = os.environ['LUCID_INPUT_RUN_UID']

merged_tags = list(dict.fromkeys(
    list(input_blob.get('tags') or []) + list(output_tags)
))

client = from_uri(
    os.environ['TILED_URL'], api_key=os.environ['TILED_API_KEY'],
)

output_uid = str(uuid.uuid4())
run = client.create_container(
    key=output_uid,
    metadata={
        'start': {
            'uid': output_uid,
            'parent_run_uid': input_uid,
            'pipeline_provenance': get_provenance(),
            'tiled_access_tags': merged_tags,
        },
    },
    access_tags=merged_tags,
)
run.write_array(signal, key='signal')
print(f'wrote output run {output_uid}')

In [ ]:
sb.glue('output_run_uids', [output_uid])
sb.glue('metrics', {'input_shape': list(signal.shape)})